In [1]:
from pathlib import Path
from datetime import date
from urllib.parse import urlsplit
import pandas as pd

# Works from the project folder or notebooks folder.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

columns = [
    "research_id", "university", "title", "authors",
    "publication_year", "publication_date", "abstract",
    "research_field", "tech_category", "journal",
    "doi", "url", "source"
]

required = [
    "research_id", "university", "title", "authors",
    "publication_year", "doi", "url", "source"
]

rana_path = root / "data/processed/final_rana.csv"
saad_path = (
    root / "data/interim/rana_saad_validation"
    / "four_sources_20260913T112108887260Z/validated.csv"
)

for path in [rana_path, saad_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Input file not found: {path}")

rana = pd.read_csv(rana_path, dtype="string")
saad = pd.read_csv(saad_path, dtype="string")

for name, frame in [("Rana Ayman", rana), ("Rana Saad", saad)]:
    missing_columns = sorted(set(columns) - set(frame.columns))
    if missing_columns:
        raise ValueError(f"{name}: missing columns {missing_columns}")

ksu = saad.loc[
    saad["university"].str.strip().eq("KSU")
].copy()

if rana.empty or ksu.empty:
    raise ValueError("One of the input datasets is empty.")

final = pd.concat(
    [rana[columns], ksu[columns]],
    ignore_index=True
)

# Standardize text and missing values.
missing_markers = {"", "nan", "none", "null", "n/a", "na", "<na>", "nat"}

for column in columns:
    text = final[column].str.strip()
    final[column] = text.mask(
        text.str.lower().isin(missing_markers),
        pd.NA
    )

final["doi"] = final["doi"].str.lower()

# Check mandatory fields before saving.
missing_counts = final[required].isna().sum()
if missing_counts.any():
    raise ValueError(
        f"Missing mandatory values:\n{missing_counts[missing_counts > 0]}"
    )

years = pd.to_numeric(final["publication_year"], errors="coerce")
valid_year = (
    years.notna()
    & years.between(2023, 2026)
    & years.mod(1).eq(0)
)
if not valid_year.all():
    raise ValueError("Invalid publication years found.")

final["publication_year"] = years.astype("Int64")

if not final["doi"].str.fullmatch(r"10\.\d{4,9}/\S+", na=False).all():
    raise ValueError("Invalid DOI format found.")

def valid_url(value):
    try:
        parts = urlsplit(value)
        return (
            parts.scheme.lower() in {"http", "https"}
            and bool(parts.hostname)
            and not any(char.isspace() for char in value)
        )
    except (ValueError, TypeError):
        return False

if not final["url"].map(valid_url).all():
    raise ValueError("Invalid URL found.")

def valid_date(value):
    if pd.isna(value):
        return True
    try:
        return date.fromisoformat(value).isoformat() == value
    except (ValueError, TypeError):
        return False

if not final["publication_date"].map(valid_date).all():
    raise ValueError("Dates must be valid YYYY-MM-DD values.")

if final["research_id"].duplicated().any():
    raise ValueError("Duplicate research_id found.")

# A research paper can belong to more than one university.
if final.duplicated(["university", "doi"]).any():
    raise ValueError("Duplicate DOI within the same university found.")

shared = final.loc[
    final["doi"].duplicated(keep=False)
].sort_values(["doi", "university"])

output_dir = root / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)

final.to_csv(
    output_dir / "final.csv",
    index=False,
    encoding="utf-8-sig"
)

shared.to_csv(
    output_dir / "shared_doi_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_dir / "final.csv")
print("Rows:", len(final))
print("Columns:", len(final.columns))
print("Unique DOIs:", final["doi"].nunique())
print("\nRows by university:")
print(final["university"].value_counts())

Saved: c:\Users\nawaf\saudi-tech-research\data\processed\final.csv
Rows: 1600
Columns: 13
Unique DOIs: 1599

Rows by university:
university
KSU      1477
KAUST     123
Name: count, dtype: Int64
